In [2]:
import json
import os
import pandas as pd

data_root = "../data/bbl_json"
data_file = "524915.json"

data_path = os.path.join(data_root, data_file)

# Load a sample JSON file and inspect it
with open(data_path, 'r') as file:
    data = json.load(file)

data['meta']
print(json.dumps(data['info'], indent=2))
print(json.dumps(data['info']['outcome'], indent=2))
data['info']['outcome']['winner']
data['info']['player_of_match']
data['innings'][0].keys()
print(json.dumps(data['innings'][0]['team'], indent=2))
print(json.dumps(data['innings'][0]['overs'], indent=2))
print(json.dumps(data['innings'][0]['overs'][0]['over'], indent=2))
print(json.dumps(data['innings'][0]['overs'][0]['deliveries'], indent=2))
print(json.dumps(data['innings'][0]['overs'][0]['deliveries'][0], indent=2))
print(json.dumps(data['innings'][0]['powerplays'], indent=2))

# Create a DataFrame to store match information
matches = pd.DataFrame(
    columns=['match_id', 'date', 'team1', 'team2', 'winner', 'result', 'winner_by_runs', 'winner_by_wickets', 'venue',
             'toss_winner', 'toss_decision', 'home_team']
)

def clean_venues(venue):
    if venue == 'Aurora Stadium':
        return 'Aurora Stadium, Launceston'
    elif venue == 'Brisbane Cricket Ground':
        return 'Brisbane Cricket Ground, Woolloongabba, Brisbane'
    elif venue == 'Brisbane Cricket Ground, Woolloongabba':
        return 'Brisbane Cricket Ground, Woolloongabba, Brisbane'
    elif venue == 'Bellerive Oval':
        return 'Bellerive Oval, Hobart'
    elif venue == 'Docklands Stadium':
        return 'Docklands Stadium, Melbourne'
    elif venue == 'Geelong Cricket Ground':
        return 'GMHBA Stadium, South Geelong, Victoria'
    elif venue == 'Simonds Stadium, South Geelong, Victoria':
        return 'GMHBA Stadium, South Geelong, Victoria'
    elif venue == 'International Sports Stadium':
        return 'International Sports Stadium, Coffs Harbour'
    elif venue == 'Manuka Oval':
        return 'Manuka Oval, Canberra'
    elif venue == 'W.A.C.A. Ground':
        return 'Western Australia Cricket Association Ground'
    elif venue == 'Sydney Cricket Ground':
        return 'Sydney Cricket Ground, Sydney'
    else:
        return venue
    
def get_home_team(info):
    venue = info['venue']
    teams = info['teams']
    if venue == 'Adelaide Oval' and 'Adelaide Strikers' in teams:
        return 'Adelaide Strikers'
    elif venue == 'Aurora Stadium, Launceston' and 'Hobart Hurricanes' in teams:
        return 'Hobart Hurricanes'
    elif venue == 'Bellerive Oval, Hobart' and 'Hobart Hurricanes' in teams:
        return 'Hobart Hurricanes'
    elif venue == 'Brisbane Cricket Ground, Woolloongabba, Brisbane' and 'Brisbane Heat' in teams:
        return 'Brisbane Heat'
    elif venue == 'Docklands Stadium, Melbourne' and 'Melbourne Renegades' in teams:
        return 'Melbourne Renegades'
    elif venue == 'Carrara Oval' and 'Brisbane Heat' in teams:
        return 'Brisbane Heat'
    elif venue == 'GMHBA Stadium, South Geelong, Victoria' and 'Melbourne Renegades' in teams:
        return 'Melbourne Renegades'
    elif venue == 'Melbourne Cricket Ground' and 'Melbourne Stars' in teams:
        return 'Melbourne Stars'
    elif venue == 'Melbourne Cricket Ground' and 'Melbourne Renegades' in teams:
        return 'Melbourne Renegades'
    elif venue == 'Perth Stadium' and 'Perth Scorchers' in teams:
        return 'Perth Scorchers'
    elif venue == 'Stadium Australia' and 'Sydney Sixers' in teams:
        return 'Sydney Sixers'
    elif venue == 'Stadium Australia' and 'Sydney Thunder' in teams:
        return 'Sydney Thunder'
    elif venue == 'Sydney Cricket Ground, Sydney' and 'Sydney Sixers' in teams:
        return 'Sydney Sixers'
    elif venue == 'Sydney Cricket Ground, Sydney' and 'Sydney Thunder' in teams:
        return 'Sydney Thunder'
    elif venue == 'Stadium Australia' and 'Sydney Thunder' in teams:
        return 'Sydney Thunder'
    elif venue == 'Stadium Australia' and 'Sydney Sixers' in teams:
        return 'Sydney Sixers'
    elif venue == 'Sydney Showground Stadium' and 'Sydney Thunder' in teams:
        return 'Sydney Thunder'
    elif venue == 'Sydney Showground Stadium' and 'Sydney Sixers' in teams:
        return 'Sydney Sixers'
    elif venue == 'Manuka Oval, Canberra' and 'Sydney Thunder' in teams:
        return 'Sydney Thunder'
    elif venue == 'Manuka Oval, Canberra' and 'Sydney Sixers' in teams:
        return 'Sydney Sixers'
    elif venue == 'Western Australia Cricket Association Ground' and 'Perth Scorchers' in teams:
        return 'Perth Scorchers'
    elif venue == 'University of Tasmania Stadium, Launceston' and 'Hobart Hurricanes' in teams:
        return 'Hobart Hurricanes'
    else:
        return None

# Process match files and extract match data
files = os.listdir(data_root)
for file in files:
    try:
        if file.endswith('.json'):
            with open(os.path.join(data_root, file), 'r') as f:
                data = json.load(f)
                data['info']['venue'] = clean_venues(data['info']['venue'])
                row = {
                    'match_id': file.split('.')[0], 
                    'date': data['info']['dates'][0],
                    "team1": data['info']['teams'][0],
                    "team2": data['info']['teams'][1],
                    'winner': data['info']['outcome'].get('winner', None),
                    'result': data['info']['outcome'].get('result', None),
                    'winner_by_runs': data['info']['outcome'].get('by', {}).get('runs', None),
                    'winner_by_wickets': data['info']['outcome'].get('by', {}).get('wickets', None),
                    "venue": data['info']['venue'],
                    "toss_winner": data['info']['toss'].get('winner', None),
                    "toss_decision": data['info']['toss'].get('decision', None),
                    "home_team": get_home_team(data['info'])
                }
                matches.loc[len(matches)] = row
    except Exception as e:
        print(f"Error processing {file}: {e}")

# Check some entries
venue_counts = matches['venue'].value_counts()
venue_counts
home_team_counts = matches['home_team'].value_counts()
home_team_counts

col_names = ['team1', 'team2', 'venue', 'home_team']
matches[matches['home_team'].isna()][col_names]

# Create a DataFrame for ball-by-ball data
balls_template = pd.DataFrame(
    columns=['match_id', 'date', 'team1', 'team2', 'home_team', 'winner', 'result', 'winner_by_runs', 'winner_by_wickets',
             'innings', 'team', 'over', 'delivery', 'batter',
             'bowler', 'runs_batter', 'runs_total', 'wicket', 'wicket_type', 'wicket_player_out', 'powerplay', 'powerplay_type', 'venue']
)
# Read in a a list of data frames, one for each match and concatenate after because its faster
matches_balls = []

files = os.listdir(data_root)
file_count = 0
for file in files:
    file_count += 1
    print(f"Processing file {file_count} of {len(files)}: {file}")
    try:
        if file.endswith('.json'):
            with open(os.path.join(data_root, file), 'r') as f:
                data = json.load(f)
                match_balls = balls_template.copy()
                data['info']['venue'] = clean_venues(data['info']['venue'])
                row_meta = {
                    'match_id': file.split('.')[0],
                    'date': data['info']['dates'][0],
                    "team1": data['info']['teams'][0],
                    "team2": data['info']['teams'][1],
                    'home_team': get_home_team(data['info']),
                    'winner': data['info']['outcome'].get('winner', None),
                    'result': data['info']['outcome'].get('result', None),
                    'winner_by_runs': data['info']['outcome'].get('by', {}).get('runs', None),
                    'winner_by_wickets': data['info']['outcome'].get('by', {}).get('wickets', None),
                    "venue": data['info']['venue']
                }
                innings_count = 0
                for innings in data['innings']:
                    innings_count += 1
                    over_count = 0
                    for over in innings['overs']:
                        over_count += 1
                        delivery_count = 0
                        for delivery in over['deliveries']:
                            delivery_count += 1
                            row = row_meta.copy()
                            row['innings'] = innings_count
                            row['team'] = innings['team']
                            row['over'] = over_count
                            row['delivery'] = delivery_count
                            row['batter'] = delivery['batter']
                            row['bowler'] = delivery['bowler']
                            row['runs_batter'] = delivery.get('runs', {}).get('batter', 0)
                            row['runs_total'] = delivery.get('runs', {}).get('total', 0)
                            if 'wickets' in delivery:
                                row['wicket_type'] = delivery['wickets'][0]['kind']
                                row['wicket_player_out'] = delivery['wickets'][0]['player_out']
                                row['wicket'] = True
                            else:
                                row['wicket_type'] = None
                                row['wicket_player_out'] = None
                                row['wicket'] = False
                            if 'powerplays' in innings:
                                for powerplay in innings['powerplays']:
                                    if over_count >= powerplay['from'] and over_count <= powerplay['to']:
                                        row['powerplay'] = True
                                        row['powerplay_type'] = powerplay['type']
                                        break
                                else:
                                    row['powerplay'] = False
                                    row['powerplay_type'] = None
                            else:
                                row['powerplay'] = False
                                row['powerplay_type'] = None
                            match_balls.loc[len(match_balls)] = row
                matches_balls.append(match_balls)
    except Exception as e:
        print(f"Error processing {file}: {e}")
        raise e

balls = pd.concat(matches_balls, ignore_index=True)

# View unique values for each column and check them
for col in list(balls.columns):
    print(col, ": ")
    u = balls[col].unique()
    print("    ", u)

# Sort matches as dates 
matches = matches.sort_values(by='date')
balls = balls.sort_values(by=['date', 'innings', 'over', 'delivery'])




{
  "balls_per_over": 6,
  "dates": [
    "2011-12-16"
  ],
  "event": {
    "name": "Big Bash League"
  },
  "gender": "male",
  "match_type": "T20",
  "officials": {
    "match_referees": [
      "PL Marshall"
    ],
    "tv_umpires": [
      "IH Lock"
    ],
    "umpires": [
      "BNJ Oxenford",
      "PR Reiffel"
    ]
  },
  "outcome": {
    "by": {
      "wickets": 7
    },
    "winner": "Sydney Sixers"
  },
  "overs": 20,
  "player_of_match": [
    "BJ Haddin"
  ],
  "players": {
    "Brisbane Heat": [
      "BB McCullum",
      "ML Hayden",
      "JR Hopes",
      "CA Lynn",
      "DT Christian",
      "AW Robinson",
      "PJ Forrest",
      "CD Hartley",
      "ND Buchanan",
      "NM Hauritz",
      "AC McDermott"
    ],
    "Sydney Sixers": [
      "BJ Haddin",
      "MJ Lumb",
      "NJ Maddinson",
      "SPD Smith",
      "MC Henriques",
      "B Lee",
      "DJ Bravo",
      "EJM Cowan",
      "JR Hazlewood",
      "MA Starc",
      "SCG MacGill"
    ]
  },
  "registry"

In [3]:
# Write the cleaned DataFrame to a new CSV file
matches.to_csv('../data/cleaned_match_data.csv', index=False)
balls.to_csv('../data/cleaned_ball_by_ball_data.csv', index=False)